In [ ]:
from flask import Flask, request, jsonify
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
import sounddevice as sd
from modelscope.pipelines import pipeline
from funasr import AutoModel
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
import sounddevice as sd
from speechbrain.inference.interfaces import foreign_class
import os
import cv2
import insightface
from sklearn import preprocessing
import warnings
import numpy as np
import io
warnings.filterwarnings("ignore")
app = Flask(__name__)
def convert_results(results):
    # 假设 results 是一个包含 numpy 数组的字典或列表
    if isinstance(results, dict):
        return {key: convert_results(value) for key, value in results.items()}
    elif isinstance(results, list):
        return [convert_results(item) for item in results]
    elif isinstance(results, np.ndarray):
        return results.tolist()  # 转换 NumPy 数组为 Python 列表
    elif isinstance(results, (np.int64, np.float64)):
        return float(results) if isinstance(results, np.float64) else int(results)
    return results  # 其他类型直接返回
# 加载图像处理器和模型
processor_predict_facemoe = AutoImageProcessor.from_pretrained("D:\huggingface-download\models--dima806--facial_emotions_image_detection")
model_predict_facemoe = AutoModelForImageClassification.from_pretrained("D:\huggingface-download\models--dima806--facial_emotions_image_detection")
# 定义标签对应的中文情感
labels_predict_facemoe = {
    0: "悲伤",
    1: "厌恶",
    2: "愤怒",
    3: "中性",
    4: "恐惧",
    5: "惊喜",
    6: "快乐",
}
# 加载声纹识别模型
model_verify_audioid = pipeline(
    task='speaker-verification',
    model=r'C:\Users\LR6H\.cache\modelscope\hub\damo\speech_eres2net_sv_zh-cn_16k-common',
    model_revision='v1.0.5'
)
# 加载 ASR 模型
model_audio2txt = AutoModel(
      model=r"C:\Users\LR6H\.cache\modelscope\hub\iic\speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch", 
      model_revision="v2.0.4", 
      vad_model="fsmn-vad", vad_model_revision="v2.0.4", 
      punc_model="ct-punc-c", punc_model_revision="v2.0.4")
# 加载情绪分类模型
model_text2emo = pipeline(
    task=Tasks.text_classification, 
    model=r"C:\Users\LR6H\.cache\modelscope\hub\damo\nlp_structbert_emotion-classification_chinese-base", 
    model_revision='v1.0.0'
)
# 加载情绪识别模型
model_audio2emo = foreign_class(
    source=r"C:\Users\LR6H\.cache\huggingface\hub\models--speechbrain--emotion-recognition-wav2vec2-IEMOCAP\snapshots\117a9c3dff08be81a3628eecf6a66b547ec1659b", 
    pymodule_file="custom_interface.py", 
    classname="CustomEncoderWav2vec2Classifier"
)
def funprint(outdata):
    scores = outdata.get('scores')
    labels = outdata.get('labels')
    results = []
    for i in range(len(scores)):
        if scores[i] >= 0.1:  # 设定阈值
            results.append({'label': labels[i], 'score': scores[i]})
    return results
class FaceRecognition:
    def __init__(self, gpu_id=0, face_db='face_db', threshold=1.24, det_thresh=0.50, det_size=(640, 640)):
        self.gpu_id = gpu_id
        self.face_db = face_db
        self.threshold = threshold
        self.det_thresh = det_thresh
        self.det_size = det_size
        self.model = insightface.app.FaceAnalysis(root=r'C:\Users\LR6H\anaconda3\Lib\site-packages\insightface', allowed_modules=None, providers=['CUDAExecutionProvider'])
        self.model.prepare(ctx_id=self.gpu_id, det_thresh=self.det_thresh, det_size=self.det_size)
        self.faces_embedding = list()
        self.load_faces(self.face_db)
    # 加载人脸库中的人脸
    def load_faces(self, face_db_path):
        if not os.path.exists(face_db_path):
            os.makedirs(face_db_path)
        for root, dirs, files in os.walk(face_db_path):
            for file in files:
                input_image = cv2.imdecode(np.fromfile(os.path.join(root, file), dtype=np.uint8), 1)
                if input_image is None:
                    print(f"Warning: Unable to read {file}. Skipping this file.")
                    continue
                user_name = file.split(".")[0]
                faces = self.model.get(input_image)
                if len(faces) == 0:
                    print(f"Warning: No face detected in {file}. Skipping this file.")
                    continue
                face = faces[0]
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                self.faces_embedding.append({
                    "user_name": user_name,
                    "feature": embedding
                })
    # 人脸识别
    def recognition(self, image):
        faces = self.model.get(image)
        results = list()
        for face in faces:
            # 开始人脸识别
            embedding = np.array(face.embedding).reshape((1, -1))
            embedding = preprocessing.normalize(embedding)
            user_name = "unknown"
            for com_face in self.faces_embedding:
                r = self.feature_compare(embedding, com_face["feature"], self.threshold)
                if r:
                    user_name = com_face["user_name"]
            results.append((user_name, face.bbox))
        return results
    @staticmethod
    def feature_compare(feature1, feature2, threshold):
        diff = np.subtract(feature1, feature2)
        dist = np.sum(np.square(diff), 1)
        return dist < threshold
    def register(self, image, user_name):
        faces = self.model.get(image)
        if len(faces) != 1:
            return '图片检测不到人脸'
        # 判断人脸是否存在
        embedding = np.array(faces[0].embedding).reshape((1, -1))
        embedding = preprocessing.normalize(embedding)
        is_exits = False
        for com_face in self.faces_embedding:
            r = self.feature_compare(embedding, com_face["feature"], self.threshold)
            if r:
                is_exits = True
        if is_exits:
            return '该用户已存在'
        # 符合注册条件保存图片，同时把特征添加到人脸特征库中
        cv2.imencode('.jpg', image)[1].tofile(os.path.join(self.face_db, '%s.jpg' % user_name))
        self.faces_embedding.append({
            "user_name": user_name,
            "feature": embedding
        })
        return "success"
    # 检测人脸
    def detect(self, image):
        faces = self.model.get(image)
        results = list()
        for face in faces:
            result = dict()
            # 获取人脸属性
            result["bbox"] = np.array(face.bbox).astype(np.int32)
            result["kps"] = np.array(face.kps).astype(np.int32).tolist()

            result["det_score"] = float(face.det_score)
            if face.gender is not None:
                result["gender"] = face.gender
            if face.age is not None:
                result["age"] = face.age
            if face.embedding is not None:
                result["embedding"] = face.embedding
            
            # 进行人脸识别，获取名字信息
            if face.embedding is not None:
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                user_name = "unknown"
                for com_face in self.faces_embedding:
                    if self.feature_compare(embedding, com_face["feature"], self.threshold):
                        user_name = com_face["user_name"]
                        break
                result["name"] = user_name
            else:
                result["name"] = "unknown"
            results.append(result)
        return results
detector_face = FaceRecognition()

def face_r(frame):
    out=[]
    # 获取检测结果
    detection_results = detector_face.detect(frame)[:1]
    for result in detection_results:
        bbox = result["bbox"]
        kps = result["kps"][:2]
        det_score = result["det_score"]
        name = result["name"]
        out.append(det_score)
        out.append(name)
        out.append(kps)
        out.append(result["gender"])
    return out

#应用对象.route（'/index.heml'）对函数进行装饰，这个函数写业务逻辑
@app.route('/')
@app.route('/index')
def func():
    return 'flask application'

@app.route('/predict_facemoe', methods=['POST'])
def predict_facemoe():
    if 'image' not in request.files:
        return jsonify({'error': 'No image provided'}), 400
    file = request.files['image']
    image = Image.open(io.BytesIO(file.read()))
    # 预处理图像
    inputs = processor_predict_facemoe(images=image, return_tensors="pt")
    # 进行推理
    with torch.no_grad():
        outputs = model_predict_facemoe(**inputs)
    # 获取预测结果
    logits = outputs.logits
    predicted_class = logits.argmax(-1).item()
    # 获取预测的情感
    predicted_emotion = labels_predict_facemoe[predicted_class]
    return jsonify({'predicted_emotion': predicted_emotion})

@app.route('/verify_audioid', methods=['POST'])
def verify_audioid():
    if 'speaker1' not in request.files or 'speaker2' not in request.files:
        return jsonify({'error': 'Both audio files must be provided'}), 400
    speaker1_audio = request.files['speaker1'].read()
    speaker2_audio = request.files['speaker2'].read()
    # 将音频数据转换为 NumPy 数组
    speaker1_wav = np.frombuffer(speaker1_audio, dtype=np.int16)
    speaker2_wav = np.frombuffer(speaker2_audio, dtype=np.int16)
    # 使用模型进行声纹识别
    result = model_verify_audioid([speaker1_wav, speaker2_wav])
    return jsonify(result)

@app.route('/audio2txt', methods=['POST'])
def audio2txt():
    if 'audio' not in request.files:
        return jsonify({'error': 'No audio file provided'}), 400
    audio_file = request.files['audio'].read()
    # 使用模型进行语音转文字
    result = model_audio2txt.generate(input=audio_file, batch_size_s=300, hotword='李航凯')
#     print(result)
    return jsonify({'transcription': result})

@app.route('/classify_text2emo', methods=['POST'])
def classify_text2emo():
    if 'text' not in request.json:
        return jsonify({'error': 'No text provided'}), 400
    input_text = request.json['text']
    outdata = model_text2emo(input=input_text)
    results = funprint(outdata)
    return jsonify({'results': results})

@app.route('/classify_audio2emo', methods=['POST'])
def classify_audio2emo():
    if 'audio' not in request.files:
        return jsonify({'error': 'No audio file provided'}), 400
    audio_file = request.files['audio'].read()
    # 将音频数据转换为 NumPy 数组
    audio_data = np.frombuffer(audio_file, dtype=np.float32)
    # 将音频数据保存为临时文件
    temp_wav_path = "temp_audio.wav"
    with open(temp_wav_path, 'wb') as f:
        f.write(audio_file)
    # 使用模型进行情绪识别
    out_prob, score, index, text_lab = model_audio2emo.classify_file(temp_wav_path)
    # 将结果转换为可序列化的格式
    results = {
        'probabilities': out_prob.tolist(),  # 转换为列表
        'score': score.item(),  # 获取标量
        'index': index.item(),  # 获取标量
        'label': text_lab  # 保持原样，因为它是字符串
    }
    return jsonify({'emotion': results})
@app.route('/detect_faceid', methods=['POST'])
def detect_faceid():
    if 'image' not in request.files:
        return jsonify({'error': 'No image file provided'}), 400
    # 读取图像文件
    file = request.files['image']
    image = Image.open(io.BytesIO(file.read()))
    image_np = np.array(image)
    # 调用人脸识别函数
    results = face_r(image_np)
    print(results)
    # 将结果中的 int64 转换为 int
    results = convert_results(results)
    return jsonify({'results': results})
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5003)

2024-11-12 18:44:10,008 - modelscope - INFO - initiate model from C:\Users\LR6H\.cache\modelscope\hub\damo\speech_eres2net_sv_zh-cn_16k-common
2024-11-12 18:44:10,009 - modelscope - INFO - initiate model from location C:\Users\LR6H\.cache\modelscope\hub\damo\speech_eres2net_sv_zh-cn_16k-common.
2024-11-12 18:44:10,012 - modelscope - INFO - initialize model from C:\Users\LR6H\.cache\modelscope\hub\damo\speech_eres2net_sv_zh-cn_16k-common
2024-11-12 18:44:10,869 - modelscope - WARNING - No preprocessor field found in cfg.
2024-11-12 18:44:10,870 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2024-11-12 18:44:10,870 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': 'C:\\Users\\LR6H\\.cache\\modelscope\\hub\\damo\\speech_eres2net_sv_zh-cn_16k-common'}. trying to build by task and model information.
2024-11-12 18:44:10,871 - modelscope - WARNING - No preproce

funasr version: 1.1.6.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
New version is available: 1.1.14.
Please use the command "pip install -U funasr" to upgrade.


2024-11-12 18:44:13,980 - modelscope - INFO - Use user-specified model revision: v2.0.4
2024-11-12 18:44:14,585 - modelscope - INFO - Use user-specified model revision: v2.0.4
2024-11-12 18:44:15,852 - modelscope - INFO - initiate model from C:\Users\LR6H\.cache\modelscope\hub\damo\nlp_structbert_emotion-classification_chinese-base
2024-11-12 18:44:15,853 - modelscope - INFO - initiate model from location C:\Users\LR6H\.cache\modelscope\hub\damo\nlp_structbert_emotion-classification_chinese-base.
2024-11-12 18:44:15,855 - modelscope - INFO - initialize model from C:\Users\LR6H\.cache\modelscope\hub\damo\nlp_structbert_emotion-classification_chinese-base
2024-11-12 18:44:16,937 - modelscope - INFO - head has no _keys_to_ignore_on_load_missing
2024-11-12 18:44:17,249 - modelscope - INFO - All model checkpoint weights were used when initializing ModelForTextClassification.

2024-11-12 18:44:17,250 - modelscope - INFO - All the weights of ModelForTextClassification were initialized from th

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1'}, 'CPUExecutionProvider': {}}
find model: C:\Users\LR6H\anaconda3\Lib\site-packages\insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: 

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5003
 * Running on http://192.168.1.242:5003
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [12/Nov/2024 18:46:00] "POST /classify_text2emo HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [12/Nov/2024 18:52:33] "POST /classify_text2emo HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [12/Nov/2024 18:52:34] "POST /classify_text2emo HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [12/Nov/2024 18:52:57] "POST /classify_text2emo HTTP/1.1" 200 -
ERROR:__main__:Exception on /predict_facemoe [POST]
Traceback (most recent call last):
  File "C:\Users\LR6H\anaconda3\lib\site-packages\flask\app.py", line 1455, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\LR6H\anaconda3\lib\site-packages\flask\app.py", line 869, in full_dispatch_request
    rv = self.handle_user_excep

In [ ]:
from flask import Flask, request, jsonify
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
import io

app = Flask(__name__)

# 加载图像处理器和模型
processor = AutoImageProcessor.from_pretrained("D:\huggingface-download\models--dima806--facial_emotions_image_detection")
model = AutoModelForImageClassification.from_pretrained("D:\huggingface-download\models--dima806--facial_emotions_image_detection")

# 定义标签对应的中文情感
labels = {
    0: "悲伤",
    1: "厌恶",
    2: "愤怒",
    3: "中性",
    4: "恐惧",
    5: "惊喜",
    6: "快乐",
}

@app.route('/predict_facemoe', methods=['POST'])
def predict_facemoe():
    if 'image' not in request.files:
        return jsonify({'error': 'No image provided'}), 400

    file = request.files['image']
    image = Image.open(io.BytesIO(file.read()))

    # 预处理图像
    inputs = processor(images=image, return_tensors="pt")

    # 进行推理
    with torch.no_grad():
        outputs = model(**inputs)

    # 获取预测结果
    logits = outputs.logits
    predicted_class = logits.argmax(-1).item()

    # 获取预测的情感
    predicted_emotion = labels[predicted_class]

    return jsonify({'predicted_emotion': predicted_emotion})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.242:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Nov/2024 00:52:44] "POST /predict_facemoe HTTP/1.1" 200 -


In [ ]:
from flask import Flask, request, jsonify
import numpy as np
import sounddevice as sd
import io
from modelscope.pipelines import pipeline

app = Flask(__name__)

# 加载声纹识别模型
sv_pipeline = pipeline(
    task='speaker-verification',
    model='damo/speech_eres2net_sv_zh-cn_16k-common',
    model_revision='v1.0.5'
)

@app.route('/verify_audioid', methods=['POST'])
def verify_audioid():
    if 'speaker1' not in request.files or 'speaker2' not in request.files:
        return jsonify({'error': 'Both audio files must be provided'}), 400

    speaker1_audio = request.files['speaker1'].read()
    speaker2_audio = request.files['speaker2'].read()

    # 将音频数据转换为 NumPy 数组
    speaker1_wav = np.frombuffer(speaker1_audio, dtype=np.int16)
    speaker2_wav = np.frombuffer(speaker2_audio, dtype=np.int16)

    # 使用模型进行声纹识别
    result = sv_pipeline([speaker1_wav, speaker2_wav])

    return jsonify(result)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

2024-11-03 22:57:32,746 - modelscope - INFO - Use user-specified model revision: v1.0.5
2024-11-03 22:57:33,197 - modelscope - INFO - initiate model from C:\Users\LR6H\.cache\modelscope\hub\damo\speech_eres2net_sv_zh-cn_16k-common
2024-11-03 22:57:33,198 - modelscope - INFO - initiate model from location C:\Users\LR6H\.cache\modelscope\hub\damo\speech_eres2net_sv_zh-cn_16k-common.
2024-11-03 22:57:33,200 - modelscope - INFO - initialize model from C:\Users\LR6H\.cache\modelscope\hub\damo\speech_eres2net_sv_zh-cn_16k-common
C:\Users\LR6H\anaconda3\lib\site-packages\modelscope\models\audio\sv\ERes2Net_aug.py:345: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default va

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.242:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Nov/2024 22:57:53] "POST /verify HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2024 22:58:38] "POST /verify HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2024 22:59:37] "POST /verify HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2024 23:00:50] "POST /verify HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2024 23:01:18] "POST /verify HTTP/1.1" 200 -


In [ ]:
from flask import Flask, request, jsonify
import numpy as np
import sounddevice as sd
import io
from funasr import AutoModel

app = Flask(__name__)

# 加载 ASR 模型
model = AutoModel(model="paraformer-zh", model_revision="v2.0.4", 
                  vad_model="fsmn-vad", vad_model_revision="v2.0.4", 
                  punc_model="ct-punc-c", punc_model_revision="v2.0.4")

@app.route('/audio2txt', methods=['POST'])
def audio2txt():
    if 'audio' not in request.files:
        return jsonify({'error': 'No audio file provided'}), 400

    audio_file = request.files['audio'].read()

    # 使用模型进行语音转文字
    result = model.generate(input=audio_file, batch_size_s=300, hotword='李航凯')
    print(result)
    return jsonify({'transcription': result})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

funasr version: 1.1.6.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
New version is available: 1.1.14.
Please use the command "pip install -U funasr" to upgrade.


2024-11-04 00:02:56,477 - modelscope - INFO - Use user-specified model revision: v2.0.4
C:\Users\LR6H\anaconda3\lib\site-packages\funasr\train_utils\load_pretrained_model.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for 

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.242:5000
INFO:werkzeug:Press CTRL+C to quit
  0%|                                                                                            | 0/1 [00:00<?, ?it/s]C:\Users\LR6H\anaconda3\lib\site-packages\funasr\models\paraformer\model.py:251: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.74it/s]
{'load_data': '0.000', 'extract_feat': '0.001', 'forward': '0.366', 'batch_size': '1', 'rtf': '0.152'}, : 100%|█| 1/1 [
rtf_avg: 0.152: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.71it/s]

  0%|    

[{'key': 'rand_key_2yW4Acq9GFz6Y', 'text': '你好，我是大傻逼，我是刘吉。', 'timestamp': [[1450, 1610], [1610, 1790], [1790, 1890], [1890, 2010], [2010, 2130], [2130, 2290], [2290, 2530], [2650, 2770], [2770, 2870], [2870, 3050], [3050, 3375]]}]


In [ ]:
from flask import Flask, request, jsonify
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks

app = Flask(__name__)

# 加载情绪分类模型
semantic_cls = pipeline(
    task=Tasks.text_classification, 
    model='damo/nlp_structbert_emotion-classification_chinese-base', 
    model_revision='v1.0.0'
)

def funprint(outdata):
    scores = outdata.get('scores')
    labels = outdata.get('labels')
    results = []
    for i in range(len(scores)):
        if scores[i] >= 0.1:  # 设定阈值
            results.append({'label': labels[i], 'score': scores[i]})
    return results

@app.route('/classify_text2emo', methods=['POST'])
def classify_text2emo():
    if 'text' not in request.json:
        return jsonify({'error': 'No text provided'}), 400

    input_text = request.json['text']
    outdata = semantic_cls(input=input_text)
    results = funprint(outdata)

    return jsonify({'results': results})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

2024-11-04 00:10:58,772 - modelscope - INFO - Use user-specified model revision: v1.0.0
2024-11-04 00:10:59,082 - modelscope - INFO - initiate model from C:\Users\LR6H\.cache\modelscope\hub\damo\nlp_structbert_emotion-classification_chinese-base
2024-11-04 00:10:59,083 - modelscope - INFO - initiate model from location C:\Users\LR6H\.cache\modelscope\hub\damo\nlp_structbert_emotion-classification_chinese-base.
2024-11-04 00:10:59,086 - modelscope - INFO - initialize model from C:\Users\LR6H\.cache\modelscope\hub\damo\nlp_structbert_emotion-classification_chinese-base
2024-11-04 00:10:59,779 - modelscope - INFO - head has no _keys_to_ignore_on_load_missing
C:\Users\LR6H\anaconda3\lib\site-packages\modelscope\utils\checkpoint.py:550: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.242:5000
Press CTRL+C to quit
C:\Users\LR6H\anaconda3\lib\site-packages\transformers\modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
127.0.0.1 - - [04/Nov/2024 00:11:03] "POST /classify HTTP/1.1" 200 -
127.0.0.1 - - [04/Nov/2024 00:11:15] "POST /classify HTTP/1.1" 200 -
127.0.0.1 - - [04/Nov/2024 00:11:31] "POST /classify HTTP/1.1" 200 -
127.0.0.1 - - [04/Nov/2024 00:11:31] "POST /classify HTTP/1.1" 200 -


In [ ]:
from flask import Flask, request, jsonify
import numpy as np
import sounddevice as sd
import io
from speechbrain.inference.interfaces import foreign_class

app = Flask(__name__)

# 加载情绪识别模型
classifier = foreign_class(
    source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP", 
    pymodule_file="custom_interface.py", 
    classname="CustomEncoderWav2vec2Classifier"
)

@app.route('/classify_audio2emo', methods=['POST'])
def classify_audio2emo():
    if 'audio' not in request.files:
        return jsonify({'error': 'No audio file provided'}), 400

    audio_file = request.files['audio'].read()

    # 将音频数据转换为 NumPy 数组
    audio_data = np.frombuffer(audio_file, dtype=np.float32)

    # 将音频数据保存为临时文件
    temp_wav_path = "temp_audio.wav"
    with open(temp_wav_path, 'wb') as f:
        f.write(audio_file)

    # 使用模型进行情绪识别
    out_prob, score, index, text_lab = classifier.classify_file(temp_wav_path)

    # 将结果转换为可序列化的格式
    results = {
        'probabilities': out_prob.tolist(),  # 转换为列表
        'score': score.item(),  # 获取标量
        'index': index.item(),  # 获取标量
        'label': text_lab  # 保持原样，因为它是字符串
    }

    return jsonify({'emotion': results})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

C:\Users\LR6H\anaconda3\lib\inspect.py:746: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):
C:\Users\LR6H\anaconda3\lib\site-packages\transformers\configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
speechbrain.lobes.models.huggingface_transformers.huggingface - Wav2Vec2Model is frozen.
C:\Users\LR6H\anaconda3\lib\site-packages\speechbrain\utils\checkpoints.py:194: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the 

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.242:5000
Press CTRL+C to quit
127.0.0.1 - - [04/Nov/2024 00:33:11] "POST /classify HTTP/1.1" 200 -


In [1]:
#人脸识别相关函数
import os
import cv2
import insightface
import numpy as np
from sklearn import preprocessing

import warnings
warnings.filterwarnings("ignore")

class FaceRecognition:
    def __init__(self, gpu_id=0, face_db='face_db', threshold=1.24, det_thresh=0.50, det_size=(640, 640)):
        self.gpu_id = gpu_id
        self.face_db = face_db
        self.threshold = threshold
        self.det_thresh = det_thresh
        self.det_size = det_size
        self.model = insightface.app.FaceAnalysis(root=r'C:\Users\LR6H\anaconda3\Lib\site-packages\insightface', allowed_modules=None, providers=['CUDAExecutionProvider'])
        self.model.prepare(ctx_id=self.gpu_id, det_thresh=self.det_thresh, det_size=self.det_size)
        self.faces_embedding = list()
        self.load_faces(self.face_db)
    # 加载人脸库中的人脸
    def load_faces(self, face_db_path):
        if not os.path.exists(face_db_path):
            os.makedirs(face_db_path)
        for root, dirs, files in os.walk(face_db_path):
            for file in files:
                input_image = cv2.imdecode(np.fromfile(os.path.join(root, file), dtype=np.uint8), 1)
                if input_image is None:
                    print(f"Warning: Unable to read {file}. Skipping this file.")
                    continue
                user_name = file.split(".")[0]
                faces = self.model.get(input_image)
                if len(faces) == 0:
                    print(f"Warning: No face detected in {file}. Skipping this file.")
                    continue
                face = faces[0]
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                self.faces_embedding.append({
                    "user_name": user_name,
                    "feature": embedding
                })
    # 人脸识别
    def recognition(self, image):
        faces = self.model.get(image)
        results = list()
        for face in faces:
            # 开始人脸识别
            embedding = np.array(face.embedding).reshape((1, -1))
            embedding = preprocessing.normalize(embedding)
            user_name = "unknown"
            for com_face in self.faces_embedding:
                r = self.feature_compare(embedding, com_face["feature"], self.threshold)
                if r:
                    user_name = com_face["user_name"]
            results.append((user_name, face.bbox))
        return results
    @staticmethod
    def feature_compare(feature1, feature2, threshold):
        diff = np.subtract(feature1, feature2)
        dist = np.sum(np.square(diff), 1)
        return dist < threshold

    def register(self, image, user_name):
        faces = self.model.get(image)
        if len(faces) != 1:
            return '图片检测不到人脸'
        # 判断人脸是否存在
        embedding = np.array(faces[0].embedding).reshape((1, -1))
        embedding = preprocessing.normalize(embedding)
        is_exits = False
        for com_face in self.faces_embedding:
            r = self.feature_compare(embedding, com_face["feature"], self.threshold)
            if r:
                is_exits = True
        if is_exits:
            return '该用户已存在'
        # 符合注册条件保存图片，同时把特征添加到人脸特征库中
        cv2.imencode('.jpg', image)[1].tofile(os.path.join(self.face_db, '%s.jpg' % user_name))
        self.faces_embedding.append({
            "user_name": user_name,
            "feature": embedding
        })
        return "success"
    # 检测人脸
    def detect(self, image):
        faces = self.model.get(image)
        results = list()
        for face in faces:
            result = dict()
            # 获取人脸属性
            result["bbox"] = np.array(face.bbox).astype(np.int32)
            result["kps"] = np.array(face.kps).astype(np.int32).tolist()

            result["det_score"] = float(face.det_score)
            if face.gender is not None:
                result["gender"] = face.gender
            if face.age is not None:
                result["age"] = face.age
            if face.embedding is not None:
                result["embedding"] = face.embedding
            
            # 进行人脸识别，获取名字信息
            if face.embedding is not None:
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                user_name = "unknown"
                for com_face in self.faces_embedding:
                    if self.feature_compare(embedding, com_face["feature"], self.threshold):
                        user_name = com_face["user_name"]
                        break
                result["name"] = user_name
            else:
                result["name"] = "unknown"
            results.append(result)
        return results
detector = FaceRecognition()
def face_r(frame):
    out=[]
    # 获取检测结果
    detection_results = detector.detect(frame)[:1]
    for result in detection_results:
        bbox = result["bbox"]
        kps = result["kps"][:2]
        det_score = result["det_score"]
        name = result["name"]
        out.append(det_score)
        out.append(name)
        out.append(kps)
        out.append(result["gender"])
    return out

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1'}, 'CPUExecutionProvider': {}}
find model: C:\Users\LR6H\anaconda3\Lib\site-packages\insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: 

In [ ]:
from flask import Flask, request, jsonify
import numpy as np
from PIL import Image
import io

app = Flask(__name__)

@app.route('/detect_faceid', methods=['POST'])
def detect_faceid():
    if 'image' not in request.files:
        return jsonify({'error': 'No image file provided'}), 400

    # 读取图像文件
    file = request.files['image']
    image = Image.open(io.BytesIO(file.read()))
    image_np = np.array(image)

    # 调用人脸识别函数
    results = face_r(image_np)
    print(results)

    # 将结果中的 int64 转换为 int
    results = convert_results(results)

    return jsonify({'results': results})

def convert_results(results):
    # 假设 results 是一个包含 numpy 数组的字典或列表
    if isinstance(results, dict):
        return {key: convert_results(value) for key, value in results.items()}
    elif isinstance(results, list):
        return [convert_results(item) for item in results]
    elif isinstance(results, np.ndarray):
        return results.tolist()  # 转换 NumPy 数组为 Python 列表
    elif isinstance(results, (np.int64, np.float64)):
        return float(results) if isinstance(results, np.float64) else int(results)
    return results  # 其他类型直接返回

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.242:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Nov/2024 00:31:16] "POST /detect HTTP/1.1" 200 -


[0.7943910956382751, 'lhk (2)', [[374, 137], [428, 137]], 0]


127.0.0.1 - - [05/Nov/2024 00:31:17] "POST /detect HTTP/1.1" 200 -


[0.8439862728118896, 'lhk (2)', [[348, 135], [399, 127]], 1]


127.0.0.1 - - [05/Nov/2024 00:31:37] "POST /detect HTTP/1.1" 200 -


[0.8026680946350098, 'lj (4)', [[353, 115], [399, 111]], 1]
